# 05 · Code with Claude — Haiku vs Sonnet for real coding tasks

Contoso Outdoors' ops team keeps asking engineers for small one-off scripts: a shipping calculator, a reorder-quantity check, a quick bug fix. Claude models are strong at code — but "strong" isn't the whole story. `claude-haiku-4-5` and `claude-sonnet-4-6` both write code; they differ in how far you can push them.

**Learning objectives**
- Generate working Python from a plain-English spec with both models
- Run the generated code and check it's actually correct, not just plausible
- Give both models a harder debugging task and see where they diverge
- Walk away with a rule of thumb for picking Haiku vs Sonnet on a coding task

`~20 minutes`


## 1 · Set up and a helper to ask for code

Same `.env` as every other lab. `ask_for_code(model, prompt)` sends a plain-English spec and returns the model's reply plus which model actually served it.


In [1]:
import os
import re
import time
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient


def find_agent_builder() -> Path:
    """Locate foundry/agent-builder from anywhere in the tree (repo root or labs/more)."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "foundry" / "agent-builder" / "src").is_dir():
            return base / "foundry" / "agent-builder"
        if base.name == "agent-builder" and (base / "src").is_dir():
            return base
    raise FileNotFoundError("Run labs/core/00-validate-setup.ipynb first — src/.env not found.")


AB = find_agent_builder()
load_dotenv(AB / "src" / ".env")
project_client = AIProjectClient(
    endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
    credential=DefaultAzureCredential(),
)
openai_client = project_client.get_openai_client()


def ask_for_code(model: str, prompt: str) -> dict:
    """Ask a model to write code via the Responses API, timing the round trip.

    Responses API (not Chat Completions): the only shape Claude serves on Foundry.
    """
    start = time.perf_counter()
    resp = openai_client.responses.create(model=model, input=prompt)
    return {
        "asked_model": model,
        "served_model": resp.model,
        "answer": resp.output_text,
        "elapsed_s": round(time.perf_counter() - start, 2),
    }


def extract_code(answer: str) -> str:
    """Pull the first fenced code block out of a reply, or return the reply as-is."""
    match = re.search(r"```(?:python)?\n(.*?)```", answer, re.DOTALL)
    return match.group(1) if match else answer


print("Ready. `ask_for_code(model, prompt)` sends a spec and returns code + timing.")


Ready. `ask_for_code(model, prompt)` sends a spec and returns code + timing.


## 2 · Check Claude is deployed

This lab only works if `claude-sonnet-4-6` and `claude-haiku-4-5` are deployed in your project. Skillable pre-provisions both; self-guided learners deploy them optionally via `provision.sh`. We check before going further so you get a clear message instead of a confusing API error later.


In [2]:
REQUIRED_CLAUDE = ["claude-sonnet-4-6", "claude-haiku-4-5"]

available = {
    getattr(d, "name", None) or getattr(d, "deployment_name", None)
    for d in project_client.deployments.list()
}
available.discard(None)

missing_claude = [name for name in REQUIRED_CLAUDE if name not in available]
for name in REQUIRED_CLAUDE:
    print(f"{'✅' if name in available else '❌'} {name}")

if missing_claude:
    raise SystemExit(
        f"\nMissing: {', '.join(missing_claude)}. Deploy them with `provision.sh` "
        "(Claude is the optional prompt) or in the Foundry portal, then re-run this cell."
    )
print("\nBoth Claude models are deployed — let's use them.")


✅ claude-sonnet-4-6
✅ claude-haiku-4-5

Both Claude models are deployed — let's use them.


## 3 · Hello, Sonnet — a task that needs deep reasoning

`claude-sonnet-4-6` is built for problems with real trade-offs, not just syntax. We give it a genuinely hard systems-design question and let it think out loud.

> ❓ **Can Sonnet reason through a multi-part design problem and justify its choices, not just list options?**


In [3]:
SONNET_STRENGTH_PROMPT = """
Contoso Outdoors sells the same tent SKUs through its website and three retail
stores, from a shared warehouse. Customers are seeing "in stock" online, then
finding it sold out at checkout because a store sale drained it minutes earlier.

Design a stock-reservation approach that fixes this. Cover: how you'd prevent
overselling across channels, what happens if a reservation times out, and one
trade-off your design accepts. Keep it to a few short paragraphs, not code.
"""

hello_sonnet = ask_for_code("claude-sonnet-4-6", SONNET_STRENGTH_PROMPT)
print(f"--- {hello_sonnet['asked_model']} ({hello_sonnet['elapsed_s']}s) ---")
print(hello_sonnet["answer"])


--- claude-sonnet-4-6 (16.04s) ---
## Stock Reservation Design for Contoso Outdoors

**Preventing Overselling**

Rather than relying on a simple available-quantity field, maintain two separate counts per SKU in a central inventory service: *on-hand* (physical units in the warehouse) and *reserved* (units temporarily held by active sessions). Displayed availability is always on-hand minus reserved. When a customer adds a tent to their cart — whether through the website or a point-of-sale terminal in a store — the system immediately places a short-lived reservation, atomically decrementing the available quantity. Because all four channels talk to the same inventory service and reservations use optimistic locking or a compare-and-swap operation, two channels cannot simultaneously claim the last unit; one will receive a "no longer available" response and surface that to the customer or associate in real time.

**Reservation Timeouts**

Every reservation carries a TTL — say, 15 minutes for 

## 4 · Hello, Haiku — a task that needs speed, not depth

`claude-haiku-4-5` is built for high-volume, low-complexity work where latency and cost matter more than deep reasoning. We give it a small, well-defined task and watch the clock.

> ❓ **Is Haiku noticeably faster on something simple, without sacrificing correctness?**


In [4]:
HAIKU_STRENGTH_PROMPT = """
Write a Python regex pattern (as a raw string) that validates Contoso Outdoors
SKU codes in the format ABC-1234 (three uppercase letters, a dash, four digits).
Return only the pattern, nothing else.
"""

hello_haiku = ask_for_code("claude-haiku-4-5", HAIKU_STRENGTH_PROMPT)
print(f"--- {hello_haiku['asked_model']} ({hello_haiku['elapsed_s']}s) ---")
print(hello_haiku["answer"])

# Same task, timed against Sonnet — this is the comparison that matters for Haiku's use case.
print(f"\nFor reference, the deep-reasoning prompt above took Sonnet {hello_sonnet['elapsed_s']}s.")
print(f"This much simpler prompt took Haiku {hello_haiku['elapsed_s']}s.")


--- claude-haiku-4-5 (2.2s) ---
```python
r'^[A-Z]{3}-\d{4}$'
```

For reference, the deep-reasoning prompt above took Sonnet 16.04s.
This much simpler prompt took Haiku 2.2s.


## 5 · The use case — a shipping-cost calculator

Contoso Outdoors ships tents, packs, and cookware of very different weights and sizes. Ops wants a small function: given an order's weight (kg) and destination zone (`domestic`, `regional`, `international`), return the shipping cost, with a rush-delivery surcharge as an option.

> ❓ **Can both `claude-haiku-4-5` and `claude-sonnet-4-6` turn that spec into working Python, first try?**


In [5]:
SHIPPING_SPEC = """
Write a Python function `shipping_cost(weight_kg: float, zone: str, rush: bool = False) -> float`
for an outdoor gear retailer. Rules:
- Base rate per kg: domestic $0.50, regional $1.20, international $3.00
- Minimum charge is $4.00 regardless of weight
- If rush=True, add a 40% surcharge on top of the computed cost
- Raise ValueError for an unknown zone
Return only the function, no example usage.
"""

haiku_code = ask_for_code("claude-haiku-4-5", SHIPPING_SPEC)
sonnet_code = ask_for_code("claude-sonnet-4-6", SHIPPING_SPEC)

for result in (haiku_code, sonnet_code):
    print(f"--- {result['asked_model']} ({result['elapsed_s']}s) ---")
    print(extract_code(result["answer"]))
    print()


--- claude-haiku-4-5 (3.06s) ---
def shipping_cost(weight_kg: float, zone: str, rush: bool = False) -> float:
    """
    Calculate shipping cost for outdoor gear retailer.
    
    Args:
        weight_kg: Weight of the package in kilograms
        zone: Shipping zone ('domestic', 'regional', or 'international')
        rush: Whether to apply rush shipping surcharge (40%)
    
    Returns:
        Total shipping cost in dollars
    
    Raises:
        ValueError: If zone is not recognized
    """
    base_rates = {
        'domestic': 0.50,
        'regional': 1.20,
        'international': 3.00
    }
    
    if zone not in base_rates:
        raise ValueError(f"Unknown zone: {zone}")
    
    rate = base_rates[zone]
    cost = weight_kg * rate
    
    cost = max(cost, 4.00)
    
    if rush:
        cost *= 1.40
    
    return cost


--- claude-sonnet-4-6 (4.91s) ---
def shipping_cost(weight_kg: float, zone: str, rush: bool = False) -> float:
    """
    Calculate shipping cost f

## 6 · Don't take the code's word for it — run it

Plausible-looking code isn't the same as correct code. We `exec()` each model's function into its own namespace and run it against test cases, including the one most people forget: an unknown zone should raise, not silently return a number.


In [6]:
TEST_CASES = [
    # (weight_kg, zone, rush, expected)
    (2.0, "domestic", False, 4.00),      # below minimum -> floor to $4.00
    (10.0, "regional", False, 12.00),    # 10 * 1.20
    (5.0, "international", True, 21.00), # 5*3.00=15.00, +40% rush = 21.00
]


def run_against_tests(label: str, code: str) -> None:
    namespace: dict = {}
    try:
        exec(code, namespace)
        fn = namespace["shipping_cost"]
    except Exception as e:
        print(f"{label}: ❌ code didn't even load — {e}")
        return

    passed = 0
    for weight, zone, rush, expected in TEST_CASES:
        try:
            actual = fn(weight, zone, rush)
            ok = abs(actual - expected) < 0.01
        except Exception as e:
            actual, ok = f"raised {type(e).__name__}", False
        passed += ok
        print(f"  {zone:14} rush={rush!s:5} -> {actual}  (expected {expected})  {'✅' if ok else '❌'}")

    # The unknown-zone case is the one that separates careful specs from quick ones.
    try:
        fn(1.0, "moon", False)
        print("  unknown zone   -> did NOT raise  ❌")
    except ValueError:
        print("  unknown zone   -> raised ValueError  ✅")
        passed += 1
    except Exception as e:
        print(f"  unknown zone   -> raised {type(e).__name__}, not ValueError  ❌")

    print(f"{label}: {passed}/{len(TEST_CASES) + 1} checks passed\n")


run_against_tests("claude-haiku-4-5", extract_code(haiku_code["answer"]))
run_against_tests("claude-sonnet-4-6", extract_code(sonnet_code["answer"]))


  domestic       rush=False -> 4.0  (expected 4.0)  ✅
  regional       rush=False -> 12.0  (expected 12.0)  ✅
  international  rush=True  -> 21.0  (expected 21.0)  ✅
  unknown zone   -> raised ValueError  ✅
claude-haiku-4-5: 4/4 checks passed

  domestic       rush=False -> 4.0  (expected 4.0)  ✅
  regional       rush=False -> 12.0  (expected 12.0)  ✅
  international  rush=True  -> 21.0  (expected 21.0)  ✅
  unknown zone   -> raised ValueError  ✅
claude-sonnet-4-6: 4/4 checks passed



## 7 · Now a harder task — find the bug

A simple, well-specified function is easy for both models. Real work is messier: here's an existing Contoso Outdoors reorder-quantity script with a subtle bug buried in it. We ask both models to find and explain it — no fix yet, just diagnosis.

> ❓ **Does raising the difficulty change which model gets it right?**


In [7]:
BUGGY_SCRIPT = '''
def reorder_quantity(current_stock: int, avg_daily_sales: float, lead_time_days: int, safety_stock: int) -> int:
    """How many units to reorder so we don't stock out before the next delivery."""
    reorder_point = avg_daily_sales * lead_time_days + safety_stock
    if current_stock > reorder_point:
        return 0
    needed = reorder_point - current_stock
    return int(needed)
'''

DEBUG_PROMPT = f"""
This function is used to decide how many units of camping gear to reorder.
It's causing intermittent stockouts that ops can't explain. Find the bug and
explain in 2-3 sentences why it causes stockouts. Don't rewrite the function yet.

```python
{BUGGY_SCRIPT}
```
"""

haiku_diagnosis = ask_for_code("claude-haiku-4-5", DEBUG_PROMPT)
sonnet_diagnosis = ask_for_code("claude-sonnet-4-6", DEBUG_PROMPT)

for result in (haiku_diagnosis, sonnet_diagnosis):
    print(f"--- {result['asked_model']} ({result['elapsed_s']}s) ---")
    print(result["answer"])
    print()

# The actual bug: int() truncates instead of rounding up (math.ceil), so every
# reorder is short by a fraction of a unit — small each time, but it compounds.


--- claude-haiku-4-5 (4.02s) ---
# Bug Found

The function calculates how many units to order **up to** the reorder point, but it should calculate how many units to order **plus** the safety stock buffer. When `current_stock` drops below the reorder point, it only reorders enough to reach that point—not enough to actually maintain the safety stock level while demand continues during the lead time. This causes stockouts because the safety stock gets consumed by sales occurring during the lead time between when the order is placed and when it arrives.

--- claude-sonnet-4-6 (8.33s) ---
## Bug Identified

The bug is that `int()` **truncates (floors) the result** rather than rounding up.

```python
return int(needed)  # int(2.9) → 2, not 3
```

### Why It Causes Stockouts

When `needed` is a non-integer (which happens whenever `avg_daily_sales` is a float like `1.5` or `2.7`), truncation silently discards the fractional unit — so the reorder is consistently short by up to almost one full u

## 8 · Which one for which task?

Across every section above, a pattern held: Haiku is fast and cheap and nails well-specified, narrow tasks; Sonnet is slower and pricier but reasons through ambiguity and trade-offs. Neither is "better" — they're sized for different jobs.

| Task looks like... | Use | Why |
|---|---|---|
| A clear spec, one function, obvious inputs/outputs | `claude-haiku-4-5` | Fast and cheap; correctness doesn't need deep reasoning |
| High-volume, repetitive requests (validators, formatters, simple transforms) | `claude-haiku-4-5` | Latency and cost compound at volume |
| Debugging a subtle, multi-step logic error | `claude-sonnet-4-6` | Needs to trace cause → effect, not just pattern-match |
| System design with real trade-offs to weigh and justify | `claude-sonnet-4-6` | Depth of reasoning matters more than speed |
| Unsure which? | Start with Haiku | If the answer looks shallow or wrong, escalate to Sonnet — you'll know quickly, and you saved the cost on the easy cases |


## 🧭 Summary — so, which Claude model do you reach for?

You put both models through four tests and saw the pattern hold every time:

| What you did | What it showed |
|---|---|
| Hello Sonnet — stock-reservation design | Sonnet reasons through ambiguity and justifies trade-offs |
| Hello Haiku — SKU regex, timed against Sonnet | Haiku is markedly faster on narrow, well-specified work |
| Shipping calculator from a clear spec | Both models can nail a well-defined function |
| Reorder-quantity bug diagnosis | Harder, ambiguous reasoning is where the models are more likely to diverge |

**Rule of thumb:** default to Haiku for volume and speed; reach for Sonnet when the task has real ambiguity or trade-offs to reason through. When unsure, start cheap and escalate.

### Try it yourself
- Give Haiku the section 7 bug-diagnosis prompt and compare its answer to Sonnet's.
- Pick a real script from your own project and ask both models to review it.

### Next
➡️ This is the last of the optional deep-dive labs. Head back to the [workshop README](../../README.md) for what's next, or revisit any lab above.
